In [ ]:
'''
1. 전체적으로 보면 solutions의 양식에서 알고리즘 이름들(Implementation, Dynamic Programming 등)에 해당하는 열들이 추가된 형태의 result를 만들어서 저장하는 코드다
2. 그런데 solutions의 모든 problem_id 행들이 아니라, problems.csv에 id가 있는 행들에 대해서만 처리할 것이므로, 없는 행들은 그냥 무시하고 넘어갈 것이다(빈 행으로 두는게 아니라, 아예 행을 안 만들 거임)
3. llm(여태까지 썼던 gemma 4 31B it 모델, api키는 역시나 있다고 가정)에게 solutions의 solution열값만 준다. 이 값은 작동이 되는 특정한 파이썬 코드다. 그 코드가 어떤 알고리즘에 해당되는지 아까 코드 작성에 사용했던 알고리즘 목록들을 참고해서 각 알고리즘에 대한 true/false를 반환하도록 만든다.
4. 예를 들어 그 solution 코드가 Implementation, Dynamic Programming만 사용한 거라면 둘만 true, 나머지 열은 false가 되게 하면 된다. 각 problem_id가 다 끝나는 시점마다 result.csv를 저장한다. 3번의 재시도 끝에도 실패한 solution은 그냥 생략(없는 행으로 침)하고 넘어간다. 참고로 각 문제 당 여러 solution이 있고, 그것들은 solution_order 값으로 구분된다(1부터 1씩 증가). 각 문제에 대한 solutions들의 알고리즘 분류가 끝날 때마다 '그 문제의 id, 번째, 그 문제의 solution 총 몇 개 중에 몇 개 성공했는지, 분류 결과는 각각 무엇인지'를 print로 출력한다. solution별 소요시간도 측정해서 한 줄에 같이 출력하고, 문제별 소요시간도 취합해서 출력한다.
5. 몇 번째 문제부터 몇 번째 문제까지만 처리할지 지정하는 변수를 상단에 둔다. 이것은 problem_id가 아니라, problems.csv의 위에서부터 봤을 때 1번째~n번째를 의미하므로 id와 불일치할 수 있다.
6. Implementation의 의미에 대한 제한은 이전에 작성했던 코드의 방식을 따른다. 그외에도 '알고리즘 분류'라는 점에서 이전과 동일하기 때문에 프롬프트를 참고해서 영어로 적는다.

In [ ]:
import os
api_key = os.environ.get("GEMINI_API_KEY")

In [5]:
import os
import time
import json
import random
import pandas as pd
from google import genai
from google.genai import types

# ==========================================
# [설정값]
# ==========================================
# api_key 변수는 이미 선언되어 있다고 가정합니다. (예: api_key = "AIzaSy...")
client = genai.Client(api_key=api_key)
MODEL_NAME = 'gemma-4-31b-it'

PROBS_FILE = 'filtered_concat_translated_problems.csv'
SOLS_FILE = 'problem_solutions.csv'
OUTPUT_FILE = 'problem_algorithms_by_solutions.csv'

# [분산 처리용 범위 설정] (problems.csv 기준 상위 1번째부터)
START_IDX = 1       # 시작 문제 순서 (1-based)
END_IDX = 1800      # 끝 문제 순서

# 26개의 대상 알고리즘 리스트 (고정)
ALGORITHMS = [
    "BFS", "Backtracking", "Binary Search", "Bitmasking", "Brute Force",
    "DFS", "Dijkstra", "Dynamic Programming", "Floyd-Warshall", "Greedy",
    "Hash Table", "Heap", "Implementation", "KMP", "Kruskal", "LCA",
    "Linked List", "Parametric Search", "Prim", "Priority Queue",
    "Segment Tree", "Sliding Window", "Topological Sort", "Trie",
    "Two Pointers", "Union-Find"
]

# ==========================================
# [1. 유틸리티 함수]
# ==========================================
def classify_codes_with_llm(solutions_list, max_retries=3):
    """
    하나의 문제에 속한 파이썬 솔루션 코드들을 한 번에 LLM에 보내어,
    각 솔루션별 사용된 알고리즘을 JSON Array 형태로 반환받습니다.
    """
    algorithms_str = ", ".join([f'"{algo}"' for algo in ALGORITHMS])
    
    # 솔루션들을 문자열로 결합
    solutions_text = ""
    for sol in solutions_list:
        solutions_text += f"\n--- [Solution Order: {sol['solution_order']}] ---\n```python\n{sol['code']}\n```\n"

    prompt = f"""
You are an expert competitive programming coach and code analyzer. 
Read the following Python solution codes for a specific algorithm problem.

[Solutions to Analyze]
{solutions_text}

Your task is to analyze EACH solution code and identify the core algorithms used in it.

[CRITICAL RULES]
1. ALGORITHM LIST: You MUST select algorithms ONLY from the following exact list:
[{algorithms_str}]

2. LIMIT OF 2 ALGORITHMS PER SOLUTION: For each solution code, select AT MOST 2 core algorithms. Focus strictly on the most CORE and standard algorithms needed to solve the problem. Do not list minor or trivial steps.

3. DEFINITION OF "Implementation": "Implementation" strictly refers to "State Simulation" (e.g., directional movement, state transitions, turn-based progression, 2D grid simulation). DO NOT select "Implementation" for mere simple coding tasks or basic logic evaluation.

4. JSON FORMAT: Return the result STRICTLY as a JSON array of objects. Each object corresponds to a solution provided.
Inside each object, you MUST include the "solution_order" integer, and the algorithms used set to true.

[Example Output Format]
If you are given Solution 1 (uses BFS) and Solution 2 (uses DFS and DP), return:
[
  {{ "solution_order": 1, "BFS": true }},
  {{ "solution_order": 2, "DFS": true, "Dynamic Programming": true }}
]

Provide only the pure JSON array. No markdown tags, no explanations.
"""

    config = types.GenerateContentConfig(
        response_mime_type="application/json",
        temperature=0.1 # 코드 분석이므로 매우 낮은 온도로 설정 (환각 방지)
    )
    
    for attempt in range(1, max_retries + 1):
        try:
            response = client.models.generate_content(
                model=MODEL_NAME,
                contents=prompt,
                config=config
            )
            
            text = response.text.strip()
            
            md_marker = "`" * 3
            if text.startswith(f'{md_marker}json'):
                text = text.replace(f'{md_marker}json', '').replace(md_marker, '').strip()
            elif text.startswith(md_marker):
                text = text.replace(md_marker, '').strip()

            parsed_json = json.loads(text)
            
            if isinstance(parsed_json, list):
                return parsed_json
            else:
                raise ValueError("JSON is not a valid list.")
                
        except Exception as e:
            if attempt == max_retries:
                print(f"        ❌ [LLM 실패] 최대 재시도 횟수 초과. 에러: {e}")
                return None
                
            sleep_time = (10 * attempt) + random.uniform(1, 2)
            print(f"        ⚠️ API 에러({e}). {sleep_time:.1f}초 후 재시도합니다... ({attempt}/{max_retries})")
            time.sleep(sleep_time)

# ==========================================
# [2. 메인 파이프라인 시작]
# ==========================================
print("🚀 [Step 1] 문제 및 솔루션 데이터 로드 중...")

# 1. problems.csv 읽기 및 범위 슬라이싱
df_probs = pd.read_csv(PROBS_FILE)
start_idx = max(0, START_IDX - 1)
end_idx = min(len(df_probs), END_IDX)
target_probs = df_probs.iloc[start_idx:end_idx]
target_ids = set(target_probs['id'])

print(f"✅ 총 {len(df_probs)}문제 중 {START_IDX}번째 ~ {end_idx}번째 문제(총 {len(target_probs)}개)를 타겟팅합니다.")

# 2. solutions.csv를 청크 단위로 읽어 타겟 문제의 솔루션만 메모리에 수집 (초고속 필터링)
print("⏳ problem_solutions.csv에서 타겟 문제의 솔루션 코드를 수집 중...")
target_solutions = {pid: [] for pid in target_ids}
chunk_iter = pd.read_csv(SOLS_FILE, chunksize=10000)

for chunk in chunk_iter:
    # target_ids에 포함된 문제만 필터링
    filtered_chunk = chunk[chunk['problem_id'].isin(target_ids)]
    for _, row in filtered_chunk.iterrows():
        target_solutions[row['problem_id']].append({
            'solution_order': int(row['solution_order']),
            'code': str(row['solution'])
        })

# 3. 결과 CSV 구조 셋업 및 이미 처리된 문제 확인
csv_columns = ['problem_id', 'solution_order', 'count'] + ALGORITHMS
processed_ids = set()

if os.path.exists(OUTPUT_FILE):
    print(f"📂 기존 작업 파일({OUTPUT_FILE})을 발견하여 이어서 작업합니다.")
    df_out = pd.read_csv(OUTPUT_FILE)
    processed_ids = set(df_out['problem_id'])
else:
    print(f"📄 새로운 작업 파일({OUTPUT_FILE})을 생성합니다.\n")
    pd.DataFrame(columns=csv_columns).to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')

# ==========================================
# [3. 알고리즘 분류 루프]
# ==========================================
processed_count = 0

for table_idx, (index, row) in enumerate(target_probs.iterrows(), start=START_IDX):
    prob_id = row['id']
    sols_for_prob = target_solutions.get(prob_id, [])
    
    # [조건 1] 처리할 솔루션 코드가 아예 없는 문제는 조용히 패스
    if not sols_for_prob:
        continue
        
    # [조건 2] 이미 완료된 문제는 패스
    if prob_id in processed_ids:
        print(f"⏭ [{table_idx}번째] Problem {prob_id} 은(는) 이미 분석이 완료되어 건너뜁니다.")
        continue

    print(f"\n📝 [{table_idx}번째] Problem {prob_id} | 솔루션 {len(sols_for_prob)}개 분석 요청 중...")
    prob_start_time = time.time()
    
    # LLM API 호출 (이 문제에 달린 여러 개의 솔루션을 한 번에 던짐)
    llm_results = classify_codes_with_llm(sols_for_prob)
    
    prob_elapsed_time = time.time() - prob_start_time
    
    if llm_results:
        rows_to_save = []
        success_count = 0
        
        # LLM이 반환한 결과(배열)를 순회하며 데이터 정제
        for result_item in llm_results:
            sol_order = result_item.get('solution_order')
            if not sol_order: continue
            
            row_dict = {
                'problem_id': prob_id,
                'solution_order': sol_order
            }
            
            true_count = 0
            for algo in ALGORITHMS:
                is_used = bool(result_item.get(algo, False))
                row_dict[algo] = is_used
                if is_used:
                    true_count += 1
                    
            row_dict['count'] = true_count
            rows_to_save.append(row_dict)
            success_count += 1
            
        # CSV 실시간 Append 저장
        if rows_to_save:
            df_new_rows = pd.DataFrame(rows_to_save, columns=csv_columns)
            df_new_rows.to_csv(OUTPUT_FILE, mode='a', header=False, index=False, encoding='utf-8-sig')
        
        # 보고서(Print) 출력
        avg_sol_time = prob_elapsed_time / len(sols_for_prob)
        print(f"  -> ✅ 완료! (문제 소요시간: {prob_elapsed_time:.1f}초 | 솔루션당 약 {avg_sol_time:.1f}초)")
        print(f"  -> 📊 성공률: 총 {len(sols_for_prob)}개 중 {success_count}개 성공")
        
        for r in rows_to_save:
            used_algos = [a for a in ALGORITHMS if r[a]]
            print(f"     [Sol {r['solution_order']}] {len(used_algos)}개 알고리즘: {', '.join(used_algos)}")
            
        processed_ids.add(prob_id)
        processed_count += 1
    else:
        print(f"  -> ❌ 3회 재시도 실패로 건너뜁니다. (문제 소요시간: {prob_elapsed_time:.1f}초)")
    
    # Rate Limit 방어를 위한 대기
    time.sleep(5)

print(f"\n🎉 작업 완료! 이번 실행에서 총 {processed_count}개 문제의 정답 코드 알고리즘 분류가 저장되었습니다.")
print(f"📁 결과 저장 파일: {OUTPUT_FILE}")

🚀 [Step 1] 문제 및 솔루션 데이터 로드 중...
✅ 총 3544문제 중 1번째 ~ 1800번째 문제(총 1800개)를 타겟팅합니다.
⏳ problem_solutions.csv에서 타겟 문제의 솔루션 코드를 수집 중...
📂 기존 작업 파일(problem_algorithms_by_solutions.csv)을 발견하여 이어서 작업합니다.
⏭ [1번째] Problem 2.0 은(는) 이미 분석이 완료되어 건너뜁니다.
⏭ [2번째] Problem 11.0 은(는) 이미 분석이 완료되어 건너뜁니다.
⏭ [3번째] Problem 19.0 은(는) 이미 분석이 완료되어 건너뜁니다.
⏭ [4번째] Problem 41.0 은(는) 이미 분석이 완료되어 건너뜁니다.
⏭ [5번째] Problem 51.0 은(는) 이미 분석이 완료되어 건너뜁니다.
⏭ [6번째] Problem 53.0 은(는) 이미 분석이 완료되어 건너뜁니다.
⏭ [7번째] Problem 58.0 은(는) 이미 분석이 완료되어 건너뜁니다.
⏭ [8번째] Problem 59.0 은(는) 이미 분석이 완료되어 건너뜁니다.
⏭ [9번째] Problem 64.0 은(는) 이미 분석이 완료되어 건너뜁니다.
⏭ [10번째] Problem 73.0 은(는) 이미 분석이 완료되어 건너뜁니다.
⏭ [11번째] Problem 78.0 은(는) 이미 분석이 완료되어 건너뜁니다.
⏭ [12번째] Problem 89.0 은(는) 이미 분석이 완료되어 건너뜁니다.
⏭ [13번째] Problem 92.0 은(는) 이미 분석이 완료되어 건너뜁니다.
⏭ [14번째] Problem 96.0 은(는) 이미 분석이 완료되어 건너뜁니다.
⏭ [15번째] Problem 100.0 은(는) 이미 분석이 완료되어 건너뜁니다.
⏭ [16번째] Problem 106.0 은(는) 이미 분석이 완료되어 건너뜁니다.
⏭ [17번째] Problem 113.0 은(는) 이미 분석이 완료되어 건너뜁니다.
⏭ [18번째] Problem 116.0 은(는) 이미 분

KeyboardInterrupt: 